In [ ]:
# 업체별 다른 패턴으로 정규화 패턴으로 다시 찾기 
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import re

search_queries = [
    ("쉐이크쉑", "쉐이크쉑 신림점"),
    ("맥도날드", "맥도날드 신림점"),
    ("롯데리아", "롯데리아 신림역점"),
    ("버거킹", "버거킹 신림역점"),
    ("버거운버거", "버거운버거 서울대점"),
    ("KFC", "KFC 신림역"),
    ("노브랜드버거", "노브랜드버거 신림남부점"),
    ("프랭크버거", "프랭크버거 신림녹두거리점"),
    ("움버거앤윙스", "움버거앤윙스 신림역점"),
    ("버거리", "버거리 낙성대점")
]

# Chrome 옵션 설정
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--remote-debugging-port=9222")
options.add_argument("--disable-web-security")
options.add_argument("--allow-running-insecure-content")
options.add_argument("--disable-extensions")
options.add_argument("--disable-plugins")
options.add_argument("--disable-images")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)

# User-Agent 설정
options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

try:
    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, 20)
    
    # 봇 감지 방지 스크립트
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    
    print("✅ Chrome 브라우저가 성공적으로 시작되었습니다.")
    print(f"🔧 디버깅 포트: 9222")
    
except Exception as e:
    print(f"❌ Chrome 브라우저 시작 실패: {e}")
    exit()

total_global_count = 0
total_global_menu_details = []

def detect_store_type():
    """매장 타입 감지"""
    try:
        # 네이버 주문 시스템 (쉐이크쉑 타입)
        if driver.find_elements(By.CSS_SELECTOR, "div.naver_order_contents"):
            return "naver_order"
        
        # 매장 섹션 시스템 (맥도날드 타입)
        elif driver.find_elements(By.CSS_SELECTOR, "div.place_section_content"):
            return "place_section"
        
        # 일반 메뉴 리스트
        elif driver.find_elements(By.CSS_SELECTOR, "li.E2jtL"):
            return "menu_list"
        
        # 기타
        else:
            return "generic"
            
    except:
        return "generic"

def click_all_more_buttons():
    """모든 더보기 버튼을 클릭 - 매장 타입별 대응"""
    more_buttons_clicked = 0
    try:
        # 더보기 버튼 선택자들 (우선순위별)
        more_selectors = [
            # 네이버 주문 시스템용
            ".fvwqf[role='button']",
            "div.lfH3O.fvwqf",
            "a.fvwqf",
            "button.fvwqf",
            
            # 일반적인 더보기 버튼
            "button[aria-label='더보기']",
            ".more_btn",
            "button.more",
            "[data-testid='more-button']",
            ".order_list_more_btn",
            
            # XPath로 텍스트 기반 찾기
            "//button[contains(text(), '더보기')]",
            "//a[contains(text(), '더보기')]",
            "//div[contains(text(), '더보기')]"
        ]
        
        for selector in more_selectors:
            try:
                if selector.startswith("//"):
                    more_buttons = driver.find_elements(By.XPATH, selector)
                else:
                    more_buttons = driver.find_elements(By.CSS_SELECTOR, selector)
                
                for btn in more_buttons:
                    try:
                        if btn.is_displayed() and btn.is_enabled():
                            driver.execute_script("arguments[0].scrollIntoView(true);", btn)
                            time.sleep(0.5)
                            driver.execute_script("arguments[0].click();", btn)
                            time.sleep(2)
                            more_buttons_clicked += 1
                            print(f"   ✅ 더보기 버튼 클릭: {more_buttons_clicked}개")
                    except:
                        continue
            except:
                continue
                
    except:
        pass
    
    return more_buttons_clicked

def scroll_and_load_all_content():
    """스크롤하고 더보기 버튼을 클릭해서 모든 컨텐츠 로드"""
    print("🔄 페이지 컨텐츠 로딩 중...")
    
    max_attempts = 12
    last_height = driver.execute_script("return document.body.scrollHeight")
    
    for attempt in range(max_attempts):
        print(f"   📜 스크롤 시도 {attempt + 1}/{max_attempts}")
        
        # 점진적 스크롤
        for i in range(4):
            driver.execute_script(f"window.scrollTo(0, document.body.scrollHeight * {(i+1)/4});")
            time.sleep(1)
        
        # 더보기 버튼 클릭
        more_clicked = click_all_more_buttons()
        
        # 페이지 높이 확인
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height and more_clicked == 0:
            print("   ✅ 모든 컨텐츠 로드 완료")
            break
        
        last_height = new_height
        time.sleep(3)
    
    # 최종 스크롤
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(3)

def wait_for_page_load():
    """페이지 로딩 대기"""
    try:
        wait.until(lambda driver: driver.execute_script("return document.readyState") == "complete")
        time.sleep(3)
        
        try:
            wait.until(lambda driver: driver.execute_script("return typeof jQuery !== 'undefined' ? jQuery.active == 0 : true"))
        except:
            pass
    except:
        time.sleep(5)

def switch_to_search_iframe():
    """검색 결과 iframe으로 전환"""
    try:
        driver.switch_to.default_content()
        search_iframe = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "iframe#searchIframe")))
        driver.switch_to.frame(search_iframe)
        print("✅ 검색 iframe으로 전환")
        return True
    except Exception as e:
        print(f"❌ 검색 iframe 전환 실패: {e}")
        return False

def click_restaurant_in_list():
    """검색 결과에서 첫 번째 매장 클릭"""
    try:
        restaurant_selectors = [
            "li.UEzoS a.place_bluelink",  # 구체적인 선택자 우선
            ".place_bluelink",
            ".search_item a",
            ".item_category a",
            "a[data-cid]",
            ".place_item a"
        ]
        
        for selector in restaurant_selectors:
            try:
                restaurant_links = driver.find_elements(By.CSS_SELECTOR, selector)
                if restaurant_links:
                    first_restaurant = restaurant_links[0]
                    
                    # 요소가 보이도록 스크롤
                    driver.execute_script("arguments[0].scrollIntoView(true);", first_restaurant)
                    time.sleep(1)
                    
                    # ActionChains를 사용한 클릭
                    ActionChains(driver).move_to_element(first_restaurant).click().perform()
                    print(f"✅ 매장 클릭 성공: {selector}")
                    time.sleep(5)
                    return True
            except Exception as e:
                print(f"   매장 클릭 시도 실패 ({selector}): {e}")
                continue
        
        print("❌ 클릭 가능한 매장을 찾을 수 없습니다.")
        return False
        
    except Exception as e:
        print(f"❌ 매장 클릭 실패: {e}")
        return False

def switch_to_detail_iframe():
    """상세 정보 iframe으로 전환"""
    try:
        driver.switch_to.default_content()
        
        iframe_selectors = [
            "iframe#entryIframe",
            "iframe[name='entryIframe']",
            "iframe.place_detail_iframe"
        ]
        
        for selector in iframe_selectors:
            try:
                detail_iframe = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, selector)))
                driver.switch_to.frame(detail_iframe)
                print("✅ 상세 정보 iframe으로 전환")
                return True
            except:
                continue
        
        print("❌ 상세 정보 iframe을 찾을 수 없습니다.")
        return False
        
    except Exception as e:
        print(f"❌ 상세 정보 iframe 전환 실패: {e}")
        return False

def find_and_click_menu_tab():
    """메뉴 탭 찾기 및 클릭"""
    try:
        print("🔍 메뉴 탭 찾는 중...")
        
        # 페이지 상단으로 스크롤
        driver.execute_script("window.scrollTo(0, 0);")
        time.sleep(2)
        
        # 탭 선택자들
        tab_selectors = [
            "a._tab-menu",
            "div.YYh8o a",
            "div[class*='tab'] a",
            "nav a",
            "ul li a",
            ".flicking-camera a"
        ]
        
        all_tabs = []
        for selector in tab_selectors:
            try:
                tabs = driver.find_elements(By.CSS_SELECTOR, selector)
                if tabs:
                    all_tabs = tabs
                    print(f"   탭 요소들 찾음: {len(tabs)}개")
                    break
            except:
                continue
        
        if not all_tabs:
            print("❌ 탭 요소를 찾을 수 없습니다")
            return False
        
        # 메뉴 탭 찾기
        menu_tab = None
        for i, tab in enumerate(all_tabs):
            try:
                tab_text = tab.text.strip().lower()
                tab_href = tab.get_attribute("href") or ""
                
                print(f"   탭 {i+1}: 텍스트='{tab_text}', href='{tab_href}'")
                
                is_menu_tab = (
                    "menu" in tab_href.lower() or
                    "메뉴" in tab_text or
                    "menu" in tab_text
                )
                
                if is_menu_tab:
                    menu_tab = tab
                    print(f"   ✅ 메뉴 탭 발견: {tab_text}")
                    break
                    
            except Exception as e:
                print(f"   탭 {i+1} 확인 중 오류: {e}")
                continue
        
        # 메뉴 탭 클릭
        if menu_tab:
            try:
                driver.execute_script("arguments[0].scrollIntoView(true);", menu_tab)
                time.sleep(1)
                driver.execute_script("arguments[0].click();", menu_tab)
                time.sleep(5)
                print("✅ 메뉴 탭 클릭 완료")
                return True
                
            except Exception as e:
                print(f"❌ 메뉴 탭 클릭 실패: {e}")
                return False
        else:
            print("❌ 메뉴 탭을 찾을 수 없습니다")
            return False
            
    except Exception as e:
        print(f"❌ 메뉴 탭 찾기 실패: {e}")
        return False

def extract_menu_info_by_type(store_type):
    """매장 타입별 메뉴 정보 추출"""
    menu_details = []
    
    try:
        if store_type == "naver_order":
            menu_details = extract_naver_order_menu()
        elif store_type == "place_section":
            menu_details = extract_place_section_menu()
        elif store_type == "menu_list":
            menu_details = extract_menu_list_menu()
        else:
            menu_details = extract_generic_menu()
            
    except Exception as e:
        print(f"❌ {store_type} 타입 메뉴 추출 실패: {e}")
        
    return menu_details

def extract_naver_order_menu():
    """네이버 주문 시스템 메뉴 추출 (쉐이크쉑 타입)"""
    menu_details = []
    print("📋 네이버 주문 시스템 메뉴 추출 중...")
    
    try:
        menu_containers = driver.find_elements(By.CSS_SELECTOR, "div.naver_order_contents")
        
        for container_idx, container in enumerate(menu_containers):
            print(f"\n📦 메뉴 컨테이너 {container_idx + 1}:")
            
            food_sections = container.find_elements(By.CSS_SELECTOR, "div.order_list_inner")
            print(f"   🍽 음식 구분 수: {len(food_sections)}개")
            
            for section_idx, section in enumerate(food_sections):
                try:
                    # 카테고리 제목
                    section_title = f"카테고리 {section_idx + 1}"
                    try:
                        title_element = section.find_element(By.CSS_SELECTOR, "div.order_category_title span")
                        section_title = title_element.text.strip()
                    except:
                        pass
                    
                    # 메뉴 아이템들
                    ul_elements = section.find_elements(By.CSS_SELECTOR, "ul.order_list_area")
                    
                    for ul in ul_elements:
                        li_elements = ul.find_elements(By.CSS_SELECTOR, "li")
                        
                        for li in li_elements:
                            try:
                                menu_info = li.find_element(By.CSS_SELECTOR, "div.MenuContent__info_detail__rCviz")
                                
                                # 메뉴명
                                menu_name = menu_info.find_element(By.CSS_SELECTOR, "div.MenuContent__tit__313LA").text.strip()
                                
                                # 메뉴 설명
                                menu_desc = ""
                                try:
                                    desc_elem = menu_info.find_element(By.CSS_SELECTOR, "div.MenuContent__detail__OG635 span.detail_txt")
                                    menu_desc = desc_elem.text.strip()
                                except:
                                    pass
                                
                                # 가격
                                menu_price = "가격 없음"
                                try:
                                    price_elem = menu_info.find_element(By.CSS_SELECTOR, "div.MenuContent__price__lhCy9 strong")
                                    menu_price = price_elem.text.strip()
                                except:
                                    pass
                                
                                # 버거/세트 필터링
                                text_to_check = f"{menu_name} {menu_desc} {section_title}".lower()
                                burger_keywords = ["버거", "세트", "burger", "set", "combo", "meal"]
                                
                                if any(keyword in text_to_check for keyword in burger_keywords):
                                    menu_detail = {
                                        'category': section_title,
                                        'name': menu_name,
                                        'price': menu_price,
                                        'description': menu_desc,
                                        'type': 'naver_order'
                                    }
                                    menu_details.append(menu_detail)
                                    print(f"      ✅ 발견: {menu_name}")
                                
                            except:
                                continue
                                
                except Exception as e:
                    print(f"   ❌ 섹션 처리 오류: {e}")
                    continue
                    
    except Exception as e:
        print(f"❌ 네이버 주문 메뉴 추출 실패: {e}")
        
    return menu_details

def extract_place_section_menu():
    """매장 섹션 시스템 메뉴 추출 (맥도날드 타입)"""
    menu_details = []
    print("📋 매장 섹션 시스템 메뉴 추출 중...")
    
    try:
        # place_section_content 찾기
        section_containers = driver.find_elements(By.CSS_SELECTOR, "div.place_section_content")
        
        for container_idx, container in enumerate(section_containers):
            print(f"\n📦 섹션 컨테이너 {container_idx + 1}:")
            
            # ul > li.E2jtL 구조 찾기
            ul_elements = container.find_elements(By.CSS_SELECTOR, "ul")
            
            for ul_idx, ul in enumerate(ul_elements):
                li_elements = ul.find_elements(By.CSS_SELECTOR, "li.E2jtL")
                print(f"   📋 메뉴 아이템 수: {len(li_elements)}개")
                
                for li_idx, li in enumerate(li_elements):
                    try:
                        # 메뉴명 (div.MXkFw > span.lPzHi)
                        menu_name = ""
                        try:
                            name_container = li.find_element(By.CSS_SELECTOR, "div.MXkFw")
                            menu_name = name_container.find_element(By.CSS_SELECTOR, "span.lPzHi").text.strip()
                        except:
                            # 대체 선택자들
                            name_selectors = ["span.lPzHi", ".menu_name", "h4", "[class*='name']"]
                            for selector in name_selectors:
                                try:
                                    menu_name = li.find_element(By.CSS_SELECTOR, selector).text.strip()
                                    if menu_name:
                                        break
                                except:
                                    continue
                        
                        if not menu_name:
                            continue
                        
                        # 메뉴 설명 (div.kPogF)
                        menu_desc = ""
                        try:
                            menu_desc = li.find_element(By.CSS_SELECTOR, "div.kPogF").text.strip()
                        except:
                            # 대체 선택자들
                            desc_selectors = ["div.TRxGt", ".menu_desc", "[class*='desc']"]
                            for selector in desc_selectors:
                                try:
                                    menu_desc = li.find_element(By.CSS_SELECTOR, selector).text.strip()
                                    if menu_desc:
                                        break
                                except:
                                    continue
                        
                        # 가격 (div.GXS1X)
                        menu_price = "가격 없음"
                        try:
                            price_container = li.find_element(By.CSS_SELECTOR, "div.GXS1X")
                            # em 태그나 strong 태그에서 가격 찾기
                            price_selectors = ["em", "strong", "span"]
                            for selector in price_selectors:
                                try:
                                    price_text = price_container.find_element(By.CSS_SELECTOR, selector).text.strip()
                                    if "원" in price_text or "₩" in price_text:
                                        menu_price = price_text
                                        break
                                except:
                                    continue
                        except:
                            # 대체 가격 선택자들
                            price_selectors = ["div.GXS1X em", ".price", "strong", "[class*='price']"]
                            for selector in price_selectors:
                                try:
                                    price_text = li.find_element(By.CSS_SELECTOR, selector).text.strip()
                                    if "원" in price_text or "₩" in price_text:
                                        menu_price = price_text
                                        break
                                except:
                                    continue
                        
                        # 버거/세트 필터링
                        text_to_check = f"{menu_name} {menu_desc}".lower()
                        burger_keywords = ["버거", "세트", "burger", "set", "combo", "meal"]
                        
                        if any(keyword in text_to_check for keyword in burger_keywords):
                            menu_detail = {
                                'category': f"카테고리 {container_idx + 1}",
                                'name': menu_name,
                                'price': menu_price,
                                'description': menu_desc,
                                'type': 'place_section'
                            }
                            menu_details.append(menu_detail)
                            print(f"      ✅ 발견: {menu_name}")
                        
                    except Exception as e:
                        print(f"      ❌ 메뉴 아이템 {li_idx + 1} 처리 실패: {e}")
                        continue
                        
    except Exception as e:
        print(f"❌ 매장 섹션 메뉴 추출 실패: {e}")
        
    return menu_details

def extract_menu_list_menu():
    """메뉴 리스트 시스템 메뉴 추출"""
    menu_details = []
    print("📋 메뉴 리스트 시스템 메뉴 추출 중...")
    
    try:
        # li.E2jtL 직접 찾기
        menu_items = driver.find_elements(By.CSS_SELECTOR, "li.E2jtL")
        print(f"   📋 메뉴 아이템 수: {len(menu_items)}개")
        
        for item_idx, item in enumerate(menu_items):
            try:
                # 메뉴명
                menu_name = ""
                name_selectors = ["span.lPzHi", ".menu_name", "h4", "[class*='name']"]
                for selector in name_selectors:
                    try:
                        menu_name = item.find_element(By.CSS_SELECTOR, selector).text.strip()
                        if menu_name:
                            break
                    except:
                        continue
                
                if not menu_name:
                    continue
                
                # 메뉴 설명
                menu_desc = ""
                desc_selectors = ["div.TRxGt", "div.kPogF", ".menu_desc", "[class*='desc']"]
                for selector in desc_selectors:
                    try:
                        menu_desc = item.find_element(By.CSS_SELECTOR, selector).text.strip()
                        if menu_desc:
                            break
                    except:
                        continue
                
                # 가격
                menu_price = "가격 없음"
                price_selectors = ["div.GXS1X em", "div.GXS1X strong", ".price", "strong"]
                for selector in price_selectors:
                    try:
                        price_text = item.find_element(By.CSS_SELECTOR, selector).text.strip()
                        if "원" in price_text or "₩" in price_text:
                            menu_price = price_text
                            break
                    except:
                        continue
                
                # 버거/세트 필터링
                text_to_check = f"{menu_name} {menu_desc}".lower()
                burger_keywords = ["버거", "세트", "burger", "set", "combo", "meal"]
                
                if any(keyword in text_to_check for keyword in burger_keywords):
                    menu_detail = {
                        'category': "메뉴",
                        'name': menu_name,
                        'price': menu_price,
                        'description': menu_desc,
                        'type': 'menu_list'
                    }
                    menu_details.append(menu_detail)
                    print(f"      ✅ 발견: {menu_name}")
                
            except Exception as e:
                print(f"      ❌ 메뉴 아이템 {item_idx + 1} 처리 실패: {e}")
                continue
                
    except Exception as e:
        print(f"❌ 메뉴 리스트 메뉴 추출 실패: {e}")
        
    return menu_details

def extract_generic_menu():
    """범용 메뉴 추출 (fallback)"""
    menu_details = []
    print("📋 범용 메뉴 추출 중...")
    
    try:
        # 다양한 메뉴 컨테이너 시도
        container_selectors = [
            ".menu_container",
            ".menu_list",
            ".order_list",
            "div[class*='menu']",
            "div[class*='order']",
            "ul"
        ]
        
        menu_containers = []
        for selector in container_selectors:
            try:
                containers = driver.find_elements(By.CSS_SELECTOR, selector)
                if containers:
                    menu_containers = containers
                    print(f"   📦 컨테이너 발견: {selector}")
                    break
            except:
                continue
        
        if not menu_containers:
            print("   ❌ 메뉴 컨테이너를 찾을 수 없습니다.")
            return menu_details
        
        for container in menu_containers:
            # 메뉴 아이템들 찾기
            item_selectors = ["li", "div[class*='item']", "div[class*='menu']"]
            
            menu_items = []
            for selector in item_selectors:
                try:
                    items = container.find_elements(By.CSS_SELECTOR, selector)
                    if items:
                        menu_items = items
                        break
                except:
                    continue
            
            print(f"   📋 메뉴 아이템 수: {len(menu_items)}개")
            
            for item_idx, item in enumerate(menu_items):
                try:
                    # 메뉴명 (범용 선택자)
                    menu_name = ""
                    name_selectors = [
                        "span", "h1", "h2", "h3", "h4", "h5", "h6", 
                        "[class*='name']", "[class*='title']", ".name", ".title"
                    ]
                    
                    for selector in name_selectors:
                        try:
                            elements = item.find_elements(By.CSS_SELECTOR, selector)
                            for element in elements:
                                text = element.text.strip()
                                if text and len(text) > 2 and len(text) < 50:
                                    # 가격이 아닌 텍스트만
                                    if not re.search(r'\d+원|\₩|\$', text):
                                        menu_name = text
                                        break
                            if menu_name:
                                break
                        except:
                            continue
                    
                    if not menu_name:
                        continue
                    
                    # 가격 (패턴 기반)
                    menu_price = "가격 없음"
                    all_text_elements = item.find_elements(By.CSS_SELECTOR, "*")
                    for element in all_text_elements:
                        try:
                            text = element.text.strip()
                            # 가격 패턴 매칭
                            if re.search(r'\d{1,3}(,\d{3})*원|\₩\s*\d+|\$\s*\d+', text):
                                menu_price = text
                                break
                        except:
                            continue
                    
                    # 설명 (나머지 텍스트)
                    menu_desc = ""
                    full_text = item.text.strip()
                    if full_text:
                        lines = full_text.split('\n')
                        for line in lines:
                            line = line.strip()
                            if (line and line != menu_name and line != menu_price and 
                                len(line) > 5 and len(line) < 100 and
                                not re.search(r'\d{1,3}(,\d{3})*원|\₩|\, line')):
                                menu_desc = line
                                break
                    
                    # 버거/세트 필터링
                    text_to_check = f"{menu_name} {menu_desc}".lower()
                    burger_keywords = ["버거", "세트", "burger", "set", "combo", "meal"]
                    
                    if any(keyword in text_to_check for keyword in burger_keywords):
                        menu_detail = {
                            'category': "일반 메뉴",
                            'name': menu_name,
                            'price': menu_price,
                            'description': menu_desc,
                            'type': 'generic'
                        }
                        menu_details.append(menu_detail)
                        print(f"      ✅ 발견: {menu_name}")
                
                except Exception as e:
                    print(f"      ❌ 메뉴 아이템 {item_idx + 1} 처리 실패: {e}")
                    continue
                    
    except Exception as e:
        print(f"❌ 범용 메뉴 추출 실패: {e}")
        
    return menu_details

# 메인 실행 부분
for brand, keyword in search_queries:
    try:
        print(f"\n{'='*60}")
        print(f"🍔 {brand} 메뉴 분석 시작")
        print(f"🔍 검색어: {keyword}")
        print(f"{'='*60}")
        
        # 1. 네이버 지도 검색
        driver.get(f"https://map.naver.com/p/search/{keyword}")
        wait_for_page_load()

        # 2. 검색 결과 확인
        try:
            no_result_selectors = [
                ".place_didntmatch", 
                ".no_result",
                ".search_noResult",
                "[data-testid='no-result']"
            ]
            
            has_no_result = False
            for selector in no_result_selectors:
                if driver.find_elements(By.CSS_SELECTOR, selector):
                    has_no_result = True
                    break
            
            if has_no_result:
                print("❌ 검색 결과가 없습니다.")
                continue
        except:
            pass

        # 3. 검색 iframe으로 전환 후 매장 클릭
        if not switch_to_search_iframe():
            continue
            
        if not click_restaurant_in_list():
            continue

        # 4. 상세 정보 iframe으로 전환
        if not switch_to_detail_iframe():
            continue

        # 5. 페이지 로딩 대기
        wait_for_page_load()

        # 6. 매장 이름 확인
        try:
            store_name_selectors = [
                "span.GHAhO",
                "h1.GHAhO",
                ".place_name",
                "span.Fc1rA",
                "h1[data-testid='place-name']",
                ".place_title",
                ".store_name"
            ]
            
            store_name = "매장명 확인 불가"
            for selector in store_name_selectors:
                try:
                    name_element = driver.find_element(By.CSS_SELECTOR, selector)
                    store_name = name_element.text.strip()
                    if store_name:
                        break
                except:
                    continue
            
            print(f"🏪 매장명: {store_name}")
        except:
            print("⚠️ 매장명을 확인할 수 없습니다.")

        # 7. 메뉴 탭 찾기 및 클릭
        if not find_and_click_menu_tab():
            continue

        # 8. 메뉴 컨텐츠 로딩 대기
        wait_for_page_load()

        # 9. 매장 타입 감지
        store_type = detect_store_type()
        print(f"🔍 매장 타입 감지: {store_type}")

        # 10. 모든 컨텐츠 로드
        scroll_and_load_all_content()

        # 11. 매장 타입별 메뉴 추출
        brand_menu_details = extract_menu_info_by_type(store_type)
        
        # 12. 결과 출력
        total_brand_count = len(brand_menu_details)
        print(f"\n🎯 {brand} 총 버거/세트 메뉴 수: {total_brand_count}개")
        print(f"📋 {brand} 버거/세트 메뉴 목록:")
        
        for menu in brand_menu_details:
            # 브랜드 정보 추가
            menu['brand'] = brand
            print(f"   • {menu['name']} ({menu['category']}) - {menu['price']}")
            if menu['description']:
                print(f"     └ {menu['description']}")
            print(f"     🔧 타입: {menu['type']}")
        
        total_global_count += total_brand_count
        total_global_menu_details.extend(brand_menu_details)

    except Exception as e:
        print(f"❌ {brand} 전체 처리 오류: {e}")
        import traceback
        print(f"💡 상세 오류: {traceback.format_exc()}")

# 최종 결과 출력
print(f"\n{'='*60}")
print(f"🌟 전체 브랜드 총 버거/세트 메뉴 수: {total_global_count}개")
print(f"📊 성공한 브랜드: {len(set([menu['brand'] for menu in total_global_menu_details]))}개")
print(f"{'='*60}")

# 브랜드별 요약
brand_summary = {}
for menu in total_global_menu_details:
    brand = menu['brand']
    if brand not in brand_summary:
        brand_summary[brand] = 0
    brand_summary[brand] += 1

print("\n📊 브랜드별 버거/세트 메뉴 수 요약:")
for brand, count in brand_summary.items():
    print(f"   {brand}: {count}개")

# 매장 타입별 요약
type_summary = {}
for menu in total_global_menu_details:
    menu_type = menu.get('type', 'unknown')
    if menu_type not in type_summary:
        type_summary[menu_type] = 0
    type_summary[menu_type] += 1

print("\n🔧 매장 타입별 요약:")
for menu_type, count in type_summary.items():
    print(f"   {menu_type}: {count}개")

# 상세 결과를 파일로 저장
try:
    import json
    with open('burger_menu_results.json', 'w', encoding='utf-8') as f:
        json.dump(total_global_menu_details, f, ensure_ascii=False, indent=2)
    print("\n💾 결과가 'burger_menu_results.json' 파일로 저장되었습니다.")
except Exception as e:
    print(f"\n❌ 파일 저장 실패: {e}")

# 브라우저 종료
# try:
#     driver.quit()
#     print("\n🔒 브라우저가 안전하게 종료되었습니다.")
# except:
#     print("\n⚠️ 브라우저 종료 중 오류 발생")

# Chrome 프로세스 강제 종료 (Windows 기준)
import os
import platform

# if platform.system() == "Windows":
#     try:
#         os.system("taskkill /f /im chrome.exe /t")
#         os.system("taskkill /f /im chromedriver.exe /t")
#         print("🧹 Chrome 프로세스 정리 완료")
#     except:
#         pass


❌ Chrome 브라우저 시작 실패: Message: session not created
from disconnected: unable to connect to renderer; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
	GetHandleVerifier [0x0x7ff794f96f75+76917]
	GetHandleVerifier [0x0x7ff794f96fd0+77008]
	(No symbol) [0x0x7ff794d49dea]
	(No symbol) [0x0x7ff794d8d92b]
	(No symbol) [0x0x7ff794d8711c]
	(No symbol) [0x0x7ff794d82675]
	(No symbol) [0x0x7ff794dd5e9e]
	(No symbol) [0x0x7ff794dd5630]
	(No symbol) [0x0x7ff794dc8243]
	(No symbol) [0x0x7ff794d91431]
	(No symbol) [0x0x7ff794d921c3]
	GetHandleVerifier [0x0x7ff79526d2ad+3051437]
	GetHandleVerifier [0x0x7ff795267903+3028483]
	GetHandleVerifier [0x0x7ff79528589d+3151261]
	GetHandleVerifier [0x0x7ff794fb183e+185662]
	GetHandleVerifier [0x0x7ff794fb96ff+218111]
	GetHandleVerifier [0x0x7ff794f9faf4+112628]
	GetHandleVerifier [0x0x7ff794f9fca9+113065]
	GetHandleVerifier [0x0x7ff794f86c78+10616]
	Bas

      ✅ 발견: 한우불고기버거
      ✅ 발견: 모짜렐라 인 더 버거 베이컨
      ✅ 발견: 리아 불고기 더블(빅불)
      ✅ 발견: 더블 미라클버거
      ✅ 발견: 더블엑스투버거
      ✅ 발견: 핫크리스피치킨버거
      ✅ 발견: 리아 사각새우 더블
      ✅ 발견: 미라클버거
      ✅ 발견: 클래식치즈버거
      ✅ 발견: 리아 불고기
      ✅ 발견: 리아 새우
      ✅ 발견: 티렉스버거
      ✅ 발견: 치킨버거
      ✅ 발견: 데리버거

📦 섹션 컨테이너 4:
   📋 메뉴 아이템 수: 6개

📦 섹션 컨테이너 5:
   📋 메뉴 아이템 수: 13개

📦 섹션 컨테이너 6:
   📋 메뉴 아이템 수: 18개

📦 섹션 컨테이너 7:
   📋 메뉴 아이템 수: 8개

📦 섹션 컨테이너 8:
   📋 메뉴 아이템 수: 0개

📦 섹션 컨테이너 9:

🎯 롯데리아 총 버거/세트 메뉴 수: 57개
📋 롯데리아 버거/세트 메뉴 목록:
   • 더블 한우불고기버거 세트 (카테고리 1) - 가격 없음
     └ 국내산 한우를 사용한 패티 2장 구성으로 육즙 가득 묵직한 육풍미를 특징으로 한 프리미엄 버거
     🔧 타입: place_section
   • 모짜렐라 인 더 버거 베이컨 세트 (카테고리 1) - 가격 없음
     └ 자연산 모짜렐라 치즈와 고소한 베이컨이 만나 풍부한 맛의 버거
     🔧 타입: place_section
   • 리아 불고기 더블(빅불) 세트 (카테고리 1) - 가격 없음
     └ 불고기패티 2장으로 더 진하고 더 커진 빅 불고기버거
     🔧 타입: place_section
   • 김치불고기버거 세트 (카테고리 2) - 가격 없음
     └ 드디어 만난 불고기와 김치의 근본적 맛남 한국의 대표되는 음식을 사용하여 한국적인 컨셉을 극대화한 버거
     🔧 타입: place_section
   • 에그김치불고기버거 세트 (카테고리 2) - 가격 없음
   

      ✅ 발견: 통새우와퍼세트
      ✅ 발견: 치즈와퍼 세트
      ✅ 발견: 갈릭불고기와퍼 세트
      ✅ 발견: 와퍼 세트
      ✅ 발견: 불고기와퍼세트

📦 섹션 컨테이너 5:
   📋 메뉴 아이템 수: 7개
      ✅ 발견: 몬스터 주니어 세트
      ✅ 발견: 통새우와퍼주니어세트
      ✅ 발견: 콰트로치즈와퍼주니어세트
      ✅ 발견: 치즈와퍼주니어세트
      ✅ 발견: 와퍼주니어세트
      ✅ 발견: 불고기와퍼주니어세트
      ✅ 발견: 불맛 더블치즈버거 주니어 세트

📦 섹션 컨테이너 6:
   📋 메뉴 아이템 수: 10개
      ✅ 발견: 롱치킨버거 세트
      ✅ 발견: 더블비프불고기버거 세트
      ✅ 발견: 치킨킹 세트
      ✅ 발견: 치킨킹BLT 세트
      ✅ 발견: 비프불고기버거 세트
      ✅ 발견: 비프 & 슈림프버거 세트
      ✅ 발견: 통새우 슈림프버거 세트
      ✅ 발견: 슈림프버거 세트
      ✅ 발견: 치킨버거 세트
      ✅ 발견: 치즈버거 세트

📦 섹션 컨테이너 7:
   📋 메뉴 아이템 수: 10개
      ✅ 발견: 불맛 더블치즈앤베이컨버거
      ✅ 발견: 불맛 더블치즈버거
      ✅ 발견: 콰트로치즈와퍼 단품
      ✅ 발견: 와퍼

📦 섹션 컨테이너 8:
   📋 메뉴 아이템 수: 7개
      ✅ 발견: 불맛 더블치즈버거 주니어

📦 섹션 컨테이너 9:
   📋 메뉴 아이템 수: 10개
      ✅ 발견: 롱치킨버거
      ✅ 발견: 더블비프불고기버거
      ✅ 발견: 치킨킹
      ✅ 발견: 치킨킹BLT
      ✅ 발견: 비프불고기버거
      ✅ 발견: 비프 & 슈림프버거
      ✅ 발견: 통새우 슈림프버거
      ✅ 발견: 슈림프버거
      ✅ 발견: 치킨버거
      ✅ 발견: 치즈버거

📦 섹션 컨테이너 10:
   📋 메뉴 아이템 수: 29개
      ✅ 발견:

   ✅ 더보기 버튼 클릭: 5개
   ✅ 더보기 버튼 클릭: 6개
   ✅ 더보기 버튼 클릭: 7개
   ✅ 더보기 버튼 클릭: 8개
   ✅ 더보기 버튼 클릭: 9개
   ✅ 더보기 버튼 클릭: 10개
   📜 스크롤 시도 2/12
   ✅ 모든 컨텐츠 로드 완료
📋 매장 섹션 시스템 메뉴 추출 중...

📦 섹션 컨테이너 1:
   📋 메뉴 아이템 수: 6개
      ✅ 발견: 한입강정 SET(실속)
      ✅ 발견: [믿고먹는]한입버거운 버거 세트(세트)
      ✅ 발견: [맛찾사]달콤비프치즈베이크 세트(세트)
      ✅ 발견: 싱글통다리 세트(실속)
      ✅ 발견: [다이어트.프로틴UP]에그마요휠렛버거 세트(세트)
      ✅ 발견: [닭날개를 통으로]한입메가윙 세트(실속)

📦 섹션 컨테이너 2:
   📋 메뉴 아이템 수: 9개
      ✅ 발견: 싱글통다리 세트(실속)
      ✅ 발견: [나는둘]커플통다리 세트(실속)
      ✅ 발견: [닭날개를 통으로]한입메가윙 세트(실속)
      ✅ 발견: 버거운패밀리 세트(실속)
      ✅ 발견: 한입강정 SET(실속)
      ✅ 발견: 한입치킨 SET(실속)
      ✅ 발견: 한입순살치킨 SET(실속)
      ✅ 발견: [1인]한입치즈베이크 SET(실속)
      ✅ 발견: [잘될꺼야]하이파이브 SET

📦 섹션 컨테이너 3:
   📋 메뉴 아이템 수: 25개
      ✅ 발견: [다이어트.프로틴UP]에그마요휠렛버거 세트(세트)
      ✅ 발견: 불찡어버거 세트
      ✅ 발견: 오찡어버거 세트
      ✅ 발견: [푸짐푸짐]비프해쉬버거 세트(세트)
      ✅ 발견: [한입좌]불닭치즈카츠버거(불닭소스) 세트(세트)
      ✅ 발견: [치즈킹]블랙치즈카츠버거(블랙패퍼) 세트(세트)
      ✅ 발견: 딥치즈치킨버거 세트(세트)
      ✅ 발견: 버거운치킨버거 세트(세트)
      ✅ 발견: [믿고먹는]한입버거운 버거 세트(세트)
    

✅ 검색 iframe으로 전환
✅ 매장 클릭 성공: .place_bluelink
✅ 상세 정보 iframe으로 전환
🏪 매장명: KFC 신림역
🔍 메뉴 탭 찾는 중...
   탭 요소들 찾음: 5개
   탭 1: 텍스트='홈', href='https://pcmap.place.naver.com/restaurant/1335496966/home?entry=bmp&from=map&fromPanelNum=2&timestamp=202507120012&locale=ko&svcName=map_pcv5&searchText=KFC%20%EC%8B%A0%EB%A6%BC%EC%97%AD'
   탭 2: 텍스트='메뉴', href='https://pcmap.place.naver.com/restaurant/1335496966/menu?entry=bmp&from=map&fromPanelNum=2&timestamp=202507120012&locale=ko&svcName=map_pcv5&searchText=KFC%20%EC%8B%A0%EB%A6%BC%EC%97%AD'
   ✅ 메뉴 탭 발견: 메뉴
✅ 메뉴 탭 클릭 완료
🔍 매장 타입 감지: place_section
🔄 페이지 컨텐츠 로딩 중...
   📜 스크롤 시도 1/12
   ✅ 모든 컨텐츠 로드 완료
📋 매장 섹션 시스템 메뉴 추출 중...

📦 섹션 컨테이너 1:
   📋 메뉴 아이템 수: 4개

📦 섹션 컨테이너 2:
   📋 메뉴 아이템 수: 0개

🎯 KFC 총 버거/세트 메뉴 수: 0개
📋 KFC 버거/세트 메뉴 목록:

🍔 노브랜드버거 메뉴 분석 시작
🔍 검색어: 노브랜드버거 신림남부점
✅ 검색 iframe으로 전환
✅ 매장 클릭 성공: .place_bluelink
✅ 상세 정보 iframe으로 전환
🏪 매장명: 노브랜드버거 신림남부점
🔍 메뉴 탭 찾는 중...
   탭 요소들 찾음: 5개
   탭 1: 텍스트='홈', href='https://pcmap.place.naver.com/restaurant/1943410109